# RuleShift Notebook

## Runtime bootstrap

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

RUNTIME_SRC = Path("/kaggle/input/datasets/raptorengineer/ruleshift-runtime/src")
if not RUNTIME_SRC.is_dir():
    raise FileNotFoundError(
        "Runtime dataset not found at /kaggle/input/datasets/raptorengineer/ruleshift-runtime/src — "
        "attach the raptorengineer/ruleshift-runtime dataset."
    )
if str(RUNTIME_SRC) not in sys.path:
    sys.path.insert(0, str(RUNTIME_SRC))

## Frozen leaderboard split

In [ ]:
from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)
PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe()

## Pre-run status

In [ ]:
from IPython.display import HTML
from core.splits import MANIFEST_VERSION

_splits_loaded = sorted(frozen_splits.keys())
_episode_count = len(leaderboard_df)
_private_attached = PRIVATE_DATASET_ROOT is not None

HTML(
    '<div style="border:1px solid #ddd;border-radius:6px;padding:12px 16px;'
    'max-width:460px;font-family:monospace;font-size:13px;line-height:1.7;'
    'background:#fafafa;">'
    '<div style="font-weight:bold;margin-bottom:4px;">RuleShift &mdash; Pre-run Status</div>'
    "<table style='border-collapse:collapse;'>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Benchmark version</td><td><b>{MANIFEST_VERSION}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Episodes loaded</td><td><b>{_episode_count}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Splits available</td><td><b>{', '.join(_splits_loaded)}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Private dataset</td><td><b>{'yes' if _private_attached else 'no'}</b></td></tr>"
    "</table></div>"
)

## Official task

In [ ]:
_RULESHIFT_BINARY_DF = None


@kbench.task(
    name="ruleshift_benchmark_v1_binary_row",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
    store_task=False,
)
def _ruleshift_benchmark_v1_binary_row(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> dict:
    import json

    global _RULESHIFT_BINARY_DF

    eval_df = leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]].copy()
    binary_results = _ruleshift_benchmark_v1_binary_row.evaluate(
        llm=[llm],
        evaluation_data=eval_df,
    )
    binary_df = normalize_count_result_df(
        binary_results.as_dataframe(),
        dataframe_label="binary_df",
    )
    payload = build_kaggle_payload(binary_df=binary_df)
    validate_kaggle_payload(payload)

    _RULESHIFT_BINARY_DF = binary_df

    print("=== RuleShift Notebook — Canonical Result ===")
    print(json.dumps(payload, indent=2))
    print("=== End Canonical Result ===")
    return payload


## Official payload

In [ ]:
payload = ruleshift_benchmark_v1_binary(kbench.llm)

## Post-run result

In [ ]:
from IPython.display import HTML

_score_pct = f"{payload['score'] * 100:.1f}%"
_num = payload["numerator"]
_den = payload["denominator"]
_episodes = payload["total_episodes"]
_version = payload["benchmark_version"]

HTML(
    '<div style="border:1px solid #ddd;border-radius:6px;padding:12px 16px;'
    'max-width:460px;font-family:monospace;font-size:13px;line-height:1.7;'
    'background:#fafafa;">'
    '<div style="font-weight:bold;margin-bottom:4px;">RuleShift &mdash; Post-run Result</div>'
    "<table style='border-collapse:collapse;'>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Final score</td><td><b>{_score_pct}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Correct / Total probes</td><td><b>{_num} / {_den}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Episodes evaluated</td><td><b>{_episodes}</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Payload validation</td><td><b>Passed</b></td></tr>"
    f"<tr><td style='padding:2px 12px 2px 0;color:#555;'>Benchmark version</td><td><b>{_version}</b></td></tr>"
    "</table></div>"
)

## Analytical summary

In [ ]:
import pandas as pd
from IPython.display import HTML

_bdf = _RULESHIFT_BINARY_DF
_has_misses = (
    _bdf is not None
    and {'num_correct', 'total'}.issubset(_bdf.columns)
    and (_bdf['num_correct'] < _bdf['total']).any()
)

_TH = 'style="padding:3px 10px;text-align:left;border-bottom:1px solid #ddd;color:#555;font-weight:600;"'
_TD = 'style="padding:3px 10px;text-align:left;border-bottom:1px solid #eee;"'

if _has_misses:
    _m = _bdf[_bdf['num_correct'] < _bdf['total']].copy()
    _id_cols = [c for c in ['episode_id', 'split'] if c in _m.columns]
    _title = f'RuleShift &mdash; Episodes with misses ({len(_m)})'
    _hdrs = _id_cols + ['correct', 'total', 'missed']
    _head = '<tr>' + ''.join(f'<th {_TH}>{h}</th>' for h in _hdrs) + '</tr>'
    _body = ''
    for _, _r in _m.iterrows():
        _vals = [str(_r[c]) for c in _id_cols] + [
            str(int(_r['num_correct'])), str(int(_r['total'])),
            str(int(_r['total'] - _r['num_correct'])),
        ]
        _body += '<tr>' + ''.join(f'<td {_TD}>{v}</td>' for v in _vals) + '</tr>'
    _tbl = (
        f'<table style="border-collapse:collapse;width:100%;margin-top:4px;">'
        f'{_head}{_body}</table>'
    )
else:
    _title = 'RuleShift &mdash; Benchmark Composition'
    _rows = []
    for _sn, _feps in frozen_splits.items():
        for _fse in _feps:
            _rows.append((_sn.replace('_leaderboard', ''), _fse.episode.difficulty.name.lower()))
    _comp = pd.DataFrame(_rows, columns=['split', 'difficulty'])
    _piv = _comp.groupby('split')['difficulty'].value_counts().unstack(fill_value=0)
    _order = [c for c in ['easy', 'medium', 'hard'] if c in _piv.columns]
    _piv = _piv[[*_order, *[c for c in _piv.columns if c not in _order]]]
    _piv['total'] = _piv.sum(axis=1)
    _cols = ['split'] + list(_piv.columns)
    _head = '<tr>' + ''.join(f'<th {_TH}>{h}</th>' for h in _cols) + '</tr>'
    _body = ''
    for _idx, _row in _piv.iterrows():
        _vals = [str(_idx)] + [str(int(v)) for v in _row]
        _body += '<tr>' + ''.join(f'<td {_TD}>{v}</td>' for v in _vals) + '</tr>'
    _tbl = (
        f'<table style="border-collapse:collapse;width:100%;margin-top:4px;">'
        f'{_head}{_body}</table>'
    )

HTML(
    '<div style="border:1px solid #ddd;border-radius:6px;padding:12px 16px;'
    'max-width:620px;font-family:monospace;font-size:12px;line-height:1.6;'
    'background:#fafafa;margin-top:8px;">'
    f'<div style="font-weight:bold;margin-bottom:4px;">{_title}</div>'
    f'{_tbl}</div>'
)

## Task selection

In [ ]:
%choose ruleshift_benchmark_v1_binary